# CAI4201 Natural Language Processing: Named Entity Recognition

This notebook compares five NER approaches on the supplied CoNLL-2003 data. The implementation uses the training split for learning, validation for model selection, and the test split only for final evaluation. Exact entity-span precision, recall, and F1 are used throughout.

## 1. Project introduction

The task is to assign a BIO NER label to every token, reconstruct entity spans, and compare the predicted spans with the gold spans. The four entity types are PER, LOC, ORG, and MISC.

## 2. Imports and configuration

In [2]:
from pathlib import Path
from bisect import bisect_right
from collections import Counter
import random

from IPython.display import display

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import SGDClassifier

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
candidates = [candidate / "data" for candidate in (Path.cwd(), *Path.cwd().parents) if (candidate / "data" / "train.txt").is_file()]
if not candidates:
    candidates = [match.parent for match in Path.cwd().rglob("data/train.txt")]
if not candidates:
    raise FileNotFoundError("Could not locate data/train.txt. Open this notebook from the project folder.")
DATA_DIR = candidates[0]
ROOT = DATA_DIR.parent
RESULTS_DIR = ROOT / "results"
VISUALIZATIONS_DIR = RESULTS_DIR / "visualizations"
RESULTS_DIR.mkdir(exist_ok=True)
VISUALIZATIONS_DIR.mkdir(exist_ok=True)
RUN_EXPERIMENTS = True  # run all five experiments
ENTITY_TYPES = ["PER", "LOC", "ORG", "MISC"]

## 3. Dataset loading and exploration

Each non-empty line contains a word, POS tag, chunk tag, and NER tag. Empty lines separate sentences.

In [3]:
def read_conll(filename, keep_features=False):
    sentences = []
    current = [[], [], [], []]
    with open(DATA_DIR / filename, encoding="utf-8") as file:
        for raw_line in file:
            parts = raw_line.strip().split()
            if not parts:
                if current[0]:
                    sentences.append(tuple(current))
                    current = [[], [], [], []]
                continue
            if parts[0] == "-DOCSTART-":
                continue
            if len(parts) < 4:
                raise ValueError(f"Malformed row in {filename}: {raw_line!r}")
            for index in range(4):
                current[index].append(parts[index])
    if current[0]:
        sentences.append(tuple(current))
    return sentences if keep_features else [(tokens, ner) for tokens, _, _, ner in sentences]


train_full, valid_full, test_full = [read_conll(name, True) for name in ("train.txt", "valid.txt", "test.txt")]
train, valid, test = [[(tokens, ner) for tokens, _, _, ner in split] for split in (train_full, valid_full, test_full)]
print("sentences:", {"train": len(train), "valid": len(valid), "test": len(test)})
print("tokens:", {name: sum(len(tokens) for tokens, _ in split) for name, split in (("train", train), ("valid", valid), ("test", test))})
print("training vocabulary:", len({token for tokens, _ in train for token in tokens}))
print("maximum sentence length:", max(len(tokens) for tokens, _ in train + valid + test))

sentences: {'train': 14041, 'valid': 3250, 'test': 3453}
tokens: {'train': 203621, 'valid': 51362, 'test': 46435}
training vocabulary: 23623
maximum sentence length: 124


## 4. Data preprocessing

The checks below verify that all token-level fields remain aligned and that the supplied sentence splits are unchanged.

In [4]:
def validate_alignment(split):
    for sentence_index, (tokens, labels) in enumerate(split):
        if len(tokens) != len(labels):
            raise ValueError(f"Sentence {sentence_index} has misaligned tokens and labels")


for split in (train, valid, test):
    validate_alignment(split)
for split in (train_full, valid_full, test_full):
    for tokens, pos, chunk, ner in split:
        if not len(tokens) == len(pos) == len(chunk) == len(ner):
            raise ValueError("Token, POS, chunk, and NER lengths differ")
print("alignment checks passed")

alignment checks passed


## 5. Label encoding and BIO tagging

Labels are encoded once for the sequence models. BIO validation checks that every I-TYPE continues the same entity type.

In [5]:
tags = sorted({tag for _, labels in train for tag in labels})
tag2id = {tag: index for index, tag in enumerate(tags)}


def validate_bio(split):
    errors = []
    for sentence_index, (_, labels) in enumerate(split):
        previous_type = None
        for token_index, label in enumerate(labels):
            if label == "O":
                previous_type = None
            elif label.startswith("B-"):
                previous_type = label[2:]
            elif label.startswith("I-"):
                current_type = label[2:]
                if current_type != previous_type:
                    errors.append((sentence_index, token_index, label))
                previous_type = current_type
            else:
                errors.append((sentence_index, token_index, label))
                previous_type = None
    return errors


bio_errors = validate_bio(train)
print("labels:", tags)
print("BIO errors:", len(bio_errors))
assert not bio_errors

labels: ['B-LOC', 'B-MISC', 'B-ORG', 'B-PER', 'I-LOC', 'I-MISC', 'I-ORG', 'I-PER', 'O']
BIO errors: 0


## 6. Shared evaluation functions

An entity is correct only when its start, end, and type all match. This is span-level evaluation, not token accuracy.

In [6]:
def get_entities(labels):
    entities = set()
    start, entity_type = None, None
    for index, label in enumerate(list(labels) + ["O"]):
        starts_new = label.startswith("B-") or (label.startswith("I-") and label[2:] != entity_type)
        ends_current = label == "O" or not label.startswith("I-")
        if starts_new:
            if start is not None:
                entities.add((start, index - 1, entity_type))
            start, entity_type = index, label[2:]
        elif ends_current:
            if start is not None:
                entities.add((start, index - 1, entity_type))
            start, entity_type = None, None
    return entities


def evaluate(gold_sequences, predicted_sequences):
    if len(gold_sequences) != len(predicted_sequences):
        raise ValueError("Sentence counts differ")
    counts = {kind: {"tp": 0, "fp": 0, "fn": 0} for kind in ENTITY_TYPES}
    for gold, predicted in zip(gold_sequences, predicted_sequences):
        if len(gold) != len(predicted):
            raise ValueError("Token counts differ")
        gold_entities, predicted_entities = get_entities(gold), get_entities(predicted)
        correct = gold_entities & predicted_entities
        for _, _, kind in correct:
            counts[kind]["tp"] += 1
        for _, _, kind in predicted_entities - correct:
            if kind in counts:
                counts[kind]["fp"] += 1
        for _, _, kind in gold_entities - correct:
            if kind in counts:
                counts[kind]["fn"] += 1
    total = {key: sum(item[key] for item in counts.values()) for key in ("tp", "fp", "fn")}
    rows = []
    for kind in ENTITY_TYPES + ["Overall"]:
        values = counts.get(kind, total)
        precision = values["tp"] / (values["tp"] + values["fp"]) if values["tp"] + values["fp"] else 0.0
        recall = values["tp"] / (values["tp"] + values["fn"]) if values["tp"] + values["fn"] else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        rows.append({"entity": kind, "precision": precision, "recall": recall, "f1": f1, "support": values["tp"] + values["fn"]})
    return pd.DataFrame(rows)


## 7. Experiment 1 - N-gram model

The model uses a two-token context window, POS tags, chunk tags, capitalization, and a local bigram. An SGD classifier with logistic loss predicts a BIO tag for each token.

In [7]:
def ngram_features(tokens, pos, chunk, index):
    features = {}
    for position in range(max(0, index - 2), min(len(tokens), index + 3)):
        word = tokens[position].lower()
        if position == index:
            features[f"word={word}"] = 1
            features[f"pos={pos[index]}"] = 1
            features[f"chunk={chunk[index]}"] = 1
            features["capitalized"] = int(tokens[index][:1].isupper())
            if index:
                features[f"bigram={tokens[index - 1].lower()}_{word}"] = 1
        else:
            features[f"context{position - index}={word}"] = 1
    return features



def sgd_sparse_features(matrix):
    # DictVectorizer may produce 64-bit CSR indices. SGD needs 32-bit indices.
    if max(matrix.shape[1], matrix.nnz) > np.iinfo(np.int32).max:
        raise ValueError("N-gram feature matrix exceeds SGD's sparse index limit.")
    matrix.indices = matrix.indices.astype(np.int32, copy=False)
    matrix.indptr = matrix.indptr.astype(np.int32, copy=False)
    return matrix


def train_ngram(split):
    features = [
        ngram_features(tokens, pos, chunk, index)
        for tokens, pos, chunk, _ in split
        for index in range(len(tokens))
    ]
    labels = [label for _, _, _, sequence in split for label in sequence]
    vectorizer = DictVectorizer()
    classifier = SGDClassifier(
        loss="log_loss", alpha=1e-6, average=True, max_iter=20,
        tol=None, random_state=SEED,
    )
    classifier.fit(sgd_sparse_features(vectorizer.fit_transform(features)), labels)
    return vectorizer, classifier


def predict_ngram(vectorizer, classifier, split):
    predictions = []
    for tokens, pos, chunk, _ in split:
        features = [ngram_features(tokens, pos, chunk, index) for index in range(len(tokens))]
        matrix = sgd_sparse_features(vectorizer.transform(features))
        predictions.append(classifier.predict(matrix).tolist())
    return predictions

## 8. Experiment 2 - Word2Vec + classifier

Word2Vec is trained on training sentences only. Each token uses the concatenated vectors from itself and its immediate neighbors. Missing vectors are zero-filled and should be reported as an OOV diagnostic.

In [8]:
def embedding_context_features(model, tokens, index, lowercase=False, dimension=100):
    vectors = []
    for position in range(index - 1, index + 2):
        if position < 0 or position >= len(tokens):
            vectors.append(np.zeros(dimension, dtype=np.float32))
            continue
        key = tokens[position].lower() if lowercase else tokens[position]
        vectors.append(model[key] if key in model else np.zeros(dimension, dtype=np.float32))
    return np.concatenate(vectors)


def train_embedding_classifier(embedding_model, lowercase=False):
    x_train = [
        embedding_context_features(embedding_model, tokens, index, lowercase)
        for tokens, _ in train
        for index in range(len(tokens))
    ]
    y_train = [label for _, labels in train for label in labels]
    classifier = SGDClassifier(
        loss="log_loss", class_weight="balanced", alpha=1e-4,
        max_iter=15, tol=None, random_state=SEED
    )
    classifier.fit(np.asarray(x_train), y_train)
    return classifier


def predict_embedding_classifier(classifier, embedding_model, split, lowercase=False):
    probabilities = []
    for tokens, _ in split:
        features = [
            embedding_context_features(embedding_model, tokens, index, lowercase)
            for index in range(len(tokens))
        ]
        probabilities.append(classifier.predict_proba(np.asarray(features)))
    return probabilities


def run_word2vec_experiment():
    from gensim.models import Word2Vec
    model = Word2Vec(
        [tokens for tokens, _ in train],
        vector_size=100, window=5, min_count=2, epochs=5,
        workers=4, seed=SEED
    )
    classifier = train_embedding_classifier(model.wv)
    return model, classifier


def decode_probabilities(probabilities, labels):
    return [[labels[index] for index in scores.argmax(axis=1)] for scores in probabilities]

## 9. Experiment 3 - GloVe + classifier

GloVe uses the same context-vector classifier design as Word2Vec. The pretrained `glove-wiki-gigaword-100` model is loaded through gensim and is not trained on the assignment test data.

In [9]:
def run_glove_experiment():
    from gensim import downloader
    model = downloader.load("glove-wiki-gigaword-100")
    classifier = train_embedding_classifier(model, lowercase=True)
    return model, classifier


def report_oov_coverage(model, split, lowercase=False):
    total = sum(len(tokens) for tokens, _ in split)
    known = sum(
        1 for tokens, _ in split
        for token in tokens
        if (token.lower() if lowercase else token) in model
    )
    return {"known_tokens": known, "total_tokens": total, "oov_rate": 1 - known / total}


## 10. Experiment 4 - LSTM with pretrained embeddings

The model concatenates the Word2Vec and GloVe context vectors. Batches group similarly sized sequences and pad on the right. Sample weights exclude padding from the loss, and validation loss is used for early stopping.

In [10]:
def fixed_features(tokens, word2vec, glove):
    return np.asarray([
        np.concatenate([
            embedding_context_features(word2vec, tokens, index),
            embedding_context_features(glove, tokens, index, lowercase=True),
        ])
        for index in range(len(tokens))
    ], dtype=np.float32)


def sequence_batches(inputs, split, shuffle=False):
    import tensorflow as tf

    labels = [np.asarray([tag2id[tag] for tag in sequence], dtype=np.int32) for _, sequence in split]

    def examples():
        for values, sequence in zip(inputs, labels):
            yield values, sequence, np.ones(len(sequence), dtype=np.float32)

    input_shape = (None, *inputs[0].shape[1:])
    dataset = tf.data.Dataset.from_generator(
        examples,
        output_signature=(
            tf.TensorSpec(input_shape, tf.as_dtype(inputs[0].dtype)),
            tf.TensorSpec((None,), tf.int32),
            tf.TensorSpec((None,), tf.float32),
        ),
    )
    if shuffle:
        dataset = dataset.shuffle(len(inputs), seed=SEED)
    boundaries = [8, 16, 32, 64]
    batch_size = 512
    bucket_counts = Counter(bisect_right(boundaries, len(values)) for values in inputs)
    batch_count = sum((count + batch_size - 1) // batch_size for count in bucket_counts.values())
    batches = dataset.bucket_by_sequence_length(
        lambda values, labels, weights: tf.shape(values)[0],
        bucket_boundaries=boundaries,
        bucket_batch_sizes=[batch_size] * (len(boundaries) + 1),
    )
    return batches.apply(tf.data.experimental.assert_cardinality(batch_count)).prefetch(1)


def fit_lstm(input_layers, train_batches, valid_batches):
    import tensorflow as tf
    from tensorflow.keras.callbacks import EarlyStopping
    from tensorflow.keras.layers import Dense, LSTM

    tf.keras.utils.set_random_seed(SEED)
    model = tf.keras.Sequential([
        *input_layers,
        LSTM(64, return_sequences=True),
        Dense(len(tags), activation="softmax"),
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")
    model.fit(
        train_batches,
        validation_data=valid_batches,
        epochs=20,
        callbacks=[EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)],
        verbose=2,
    )
    return model


def predict_lstm(model, inputs, batch_size=256):
    probabilities = []
    for start in range(0, len(inputs), batch_size):
        batch = inputs[start:start + batch_size]
        max_length = max(map(len, batch))
        padded = np.zeros((len(batch), max_length, *batch[0].shape[1:]), dtype=batch[0].dtype)
        for row, values in enumerate(batch):
            padded[row, :len(values)] = values
        scores = model(padded, training=False).numpy()
        probabilities.extend(row[:len(values)] for row, values in zip(scores, batch))
    return decode_probabilities(probabilities, tags)

## 11. Experiment 5 - LSTM with trainable embedding layer

The trainable embedding model maps the training vocabulary to IDs, uses `mask_zero=True`, and predicts every original test token after removing only batch padding.

In [11]:
def build_vocabulary(split):
    counts = Counter(token for tokens, _ in split for token in tokens)
    frequent_words = (word for word, count in counts.items() if count >= 2)
    return {word: index for index, word in enumerate(frequent_words, start=2)}


def word_ids(tokens, vocabulary):
    # ID 0 is reserved for padding and ID 1 represents unknown words.
    return np.asarray([vocabulary.get(token, 1) for token in tokens], dtype=np.int32)


def train_lstm_models(word2vec, glove):
    from tensorflow.keras import Input
    from tensorflow.keras.layers import Embedding

    vocabulary = build_vocabulary(train)
    train_ids = [word_ids(tokens, vocabulary) for tokens, _ in train]
    valid_ids = [word_ids(tokens, vocabulary) for tokens, _ in valid]
    trainable_lstm = fit_lstm(
        [Input(shape=(None,), dtype="int32"), Embedding(len(vocabulary) + 2, 100, mask_zero=True)],
        sequence_batches(train_ids, train, shuffle=True),
        sequence_batches(valid_ids, valid),
    )

    train_vectors = [fixed_features(tokens, word2vec, glove) for tokens, _ in train]
    valid_vectors = [fixed_features(tokens, word2vec, glove) for tokens, _ in valid]
    pretrained_lstm = fit_lstm(
        [Input(shape=(None, train_vectors[0].shape[1]))],
        sequence_batches(train_vectors, train, shuffle=True),
        sequence_batches(valid_vectors, valid),
    )
    return vocabulary, trainable_lstm, pretrained_lstm

## 12. Final comparison of all five experiments

When `RUN_EXPERIMENTS` is enabled, each model must produce validation and test predictions, which are passed through the shared `evaluate` function. Test results are not used for model selection.

In [12]:
def wide_result_row(model, split, report):
    row = {"model": model, "split": split}
    for _, values in report.iterrows():
        prefix = "overall" if values.entity == "Overall" else values.entity.lower()
        row[f"{prefix}_precision"] = values.precision
        row[f"{prefix}_recall"] = values.recall
        row[f"{prefix}_f1"] = values.f1
    return row




def predict_all_models(full_split):
    vectorizer, ngram = ngram_model
    split = [(tokens, labels) for tokens, _, _, labels in full_split]
    tokens_by_sentence = [tokens for tokens, _ in split]
    return {
        "N-gram": predict_ngram(vectorizer, ngram, full_split),
        "Word2Vec": decode_probabilities(
            predict_embedding_classifier(word2vec_classifier, word2vec.wv, split),
            word2vec_classifier.classes_,
        ),
        "GloVe": decode_probabilities(
            predict_embedding_classifier(glove_classifier, glove, split, lowercase=True),
            glove_classifier.classes_,
        ),
        "LSTM + pretrained embeddings": predict_lstm(
            pretrained_lstm, [fixed_features(tokens, word2vec.wv, glove) for tokens in tokens_by_sentence],
        ),
        "LSTM + trainable embedding": predict_lstm(
            trainable_lstm, [word_ids(tokens, vocabulary) for tokens in tokens_by_sentence],
        ),
    }


models_ready = False
if RUN_EXPERIMENTS:
    ngram_model = train_ngram(train_full)
    word2vec, word2vec_classifier = run_word2vec_experiment()
    glove, glove_classifier = run_glove_experiment()
    vocabulary, trainable_lstm, pretrained_lstm = train_lstm_models(word2vec.wv, glove)
    models_ready = True

    rows = []
    for split_name, split in (("valid", valid_full), ("test", test_full)):
        gold = [labels for _, _, _, labels in split]
        for model_name, predictions in predict_all_models(split).items():
            rows.append(wide_result_row(model_name, split_name, evaluate(gold, predictions)))

    final_results = pd.DataFrame(rows)
    final_results.to_csv(RESULTS_DIR / "results.csv", index=False)
    final_results.to_csv(RESULTS_DIR / "per_class_results.csv", index=False)
    display(final_results[final_results.split == "test"].sort_values("overall_f1", ascending=False))
else:
    print("Set RUN_EXPERIMENTS = True to train all five models.")

Epoch 1/20
31/31 - 16s - 528ms/step - loss: 1.5734 - val_loss: 1.0581
Epoch 2/20
31/31 - 12s - 397ms/step - loss: 0.9431 - val_loss: 0.8623
Epoch 3/20
31/31 - 12s - 399ms/step - loss: 0.7898 - val_loss: 0.7324
Epoch 4/20
31/31 - 11s - 363ms/step - loss: 0.6665 - val_loss: 0.6348
Epoch 5/20
31/31 - 11s - 365ms/step - loss: 0.5623 - val_loss: 0.5614
Epoch 6/20
31/31 - 11s - 371ms/step - loss: 0.4843 - val_loss: 0.4966
Epoch 7/20
31/31 - 12s - 380ms/step - loss: 0.4199 - val_loss: 0.4434
Epoch 8/20
31/31 - 12s - 383ms/step - loss: 0.3580 - val_loss: 0.3913
Epoch 9/20
31/31 - 13s - 413ms/step - loss: 0.3016 - val_loss: 0.3463
Epoch 10/20
31/31 - 12s - 385ms/step - loss: 0.2550 - val_loss: 0.3139
Epoch 11/20
31/31 - 12s - 373ms/step - loss: 0.2168 - val_loss: 0.2880
Epoch 12/20
31/31 - 11s - 354ms/step - loss: 0.1866 - val_loss: 0.2699
Epoch 13/20
31/31 - 8s - 254ms/step - loss: 0.1624 - val_loss: 0.2527
Epoch 14/20
31/31 - 11s - 363ms/step - loss: 0.1439 - val_loss: 0.2414
Epoch 15/20
31/3

,model,split,per_precision,per_recall,per_f1,loc_precision,loc_recall,loc_f1,org_precision,org_recall,org_f1,misc_precision,misc_recall,misc_f1,overall_precision,overall_recall,overall_f1
5,N-gram,test,0.730667,0.847248,0.784651,0.811638,0.811151,0.811394,0.654135,0.680915,0.667257,0.733429,0.725071,0.729226,0.731433,0.772486,0.751399
8,LSTM + pretrained embeddings,test,0.829932,0.829932,0.829932,0.768484,0.853717,0.808861,0.650938,0.647803,0.649366,0.588319,0.588319,0.588319,0.730472,0.753364,0.741741
9,LSTM + trainable embedding,test,0.798677,0.522573,0.631776,0.729070,0.751799,0.740260,0.627273,0.498495,0.555518,0.650000,0.574074,0.609682,0.705808,0.589589,0.642485
7,GloVe,test,0.645082,0.807050,0.717033,0.609232,0.799161,0.691390,0.290746,0.586394,0.388745,0.312073,0.585470,0.407132,0.453143,0.712288,0.553903
6,Word2Vec,test,0.150000,0.484230,0.229048,0.245416,0.417266,0.309059,0.194311,0.267309,0.225038,0.039684,0.286325,0.069707,0.137868,0.376062,0.201767


Epoch 2/20


31/31 - 14s - 439ms/step - loss: 0.9431 - val_loss: 0.8623


Epoch 3/20


31/31 - 15s - 477ms/step - loss: 0.7898 - val_loss: 0.7324


... (71 outputs left out)

31/31 - 5s - 163ms/step - loss: 0.0936 - val_loss: 0.1111


Epoch 20/20


31/31 - 5s - 160ms/step - loss: 0.0905 - val_loss: 0.1063


,model,split,per_precision,per_recall,per_f1,loc_precision,loc_recall,loc_f1,org_precision,org_recall,org_f1,misc_precision,misc_recall,misc_f1,overall_precision,overall_recall,overall_f1
5,N-gram,test,0.730667,0.847248,0.784651,0.811638,0.811151,0.811394,0.654135,0.680915,0.667257,0.733429,0.725071,0.729226,0.731433,0.772486,0.751399
8,LSTM + pretrained embeddings,test,0.833230,0.828077,0.830645,0.770096,0.861511,0.813243,0.654389,0.641782,0.648024,0.589563,0.595442,0.592488,0.733092,0.754249,0.743520
9,LSTM + trainable embedding,test,0.798677,0.522573,0.631776,0.729070,0.751799,0.740260,0.627273,0.498495,0.555518,0.650000,0.574074,0.609682,0.705808,0.589589,0.642485
7,GloVe,test,0.645082,0.807050,0.717033,0.609232,0.799161,0.691390,0.290746,0.586394,0.388745,0.312073,0.585470,0.407132,0.453143,0.712288,0.553903
6,Word2Vec,test,0.148772,0.464440,0.225356,0.237821,0.421463,0.304066,0.187657,0.269115,0.221123,0.042201,0.290598,0.073699,0.138305,0.372698,0.201744


In [13]:
if models_ready:
    w2v_oov = report_oov_coverage(word2vec.wv, test, lowercase=False)
    glove_oov = report_oov_coverage(glove, test, lowercase=True)
    print(f"Word2Vec OOV rate on test: {w2v_oov['oov_rate']:.1%}")
    print(f"GloVe    OOV rate on test: {glove_oov['oov_rate']:.1%}")

Word2Vec OOV rate on test: 15.6%
GloVe    OOV rate on test: 2.9%


## 13. Visualizations

These plots use the results from the completed experiments.

In [14]:
def save_visualizations(results):
    test_results = results[results.split == "test"].sort_values("overall_f1", ascending=False)
    test_results.set_index("model")[["overall_precision", "overall_recall", "overall_f1"]].plot(kind="bar", ylim=(0, 1), figsize=(10, 5))
    plt.tight_layout()
    plt.savefig(VISUALIZATIONS_DIR / "model_performance.png", dpi=150)
    plt.close()
    test_results.set_index("model")[[f"{kind.lower()}_f1" for kind in ENTITY_TYPES]].plot(kind="bar", ylim=(0, 1), figsize=(10, 5))
    plt.tight_layout()
    plt.savefig(VISUALIZATIONS_DIR / "per_class_f1.png", dpi=150)
    plt.close()
    test_results.plot(x="model", y="overall_f1", kind="barh", legend=False, figsize=(8, 4))
    plt.tight_layout()
    plt.savefig(VISUALIZATIONS_DIR / "overall_f1_ranking.png", dpi=150)
    plt.close()

if models_ready:
    save_visualizations(final_results)

## 14. Final observations

The report should discuss the measured validation and test results, the effect of lexical context and embeddings, OOV coverage, and the difference between fixed pretrained features and a trainable embedding layer. The report and video must be written in the student's own words; this notebook supplies implementation evidence, not an AI-generated report.

## 15. Try your own text

Run the notebook from the top with `RUN_EXPERIMENTS = True`. Replace the sample in `text_to_tag` and rerun this cell for each new input. The table shows one BIO tag per token from each of the five trained models. `B-` starts an entity, `I-` continues it, and `O` marks a token outside an entity.

For raw English text, the N-gram model uses NLTK POS tags and rule-based chunk tags. Its benchmark uses the POS and chunk tags supplied with the dataset. The table displays an `I-` tag that starts a new span as `B-`, matching how the benchmark reconstructs spans.

In [15]:
def prepare_text(text):
    import nltk
    from nltk.chunk import RegexpParser, tree2conlltags

    for resource, package in (
        ("tokenizers/punkt_tab/english/", "punkt_tab"),
        ("taggers/averaged_perceptron_tagger_eng/", "averaged_perceptron_tagger_eng"),
    ):
        try:
            nltk.data.find(resource)
        except LookupError:
            nltk.download(package, quiet=True, raise_on_error=True)

    chunker = RegexpParser(r'''
        NP: {<DT|PDT|PRP\$|CD>?<JJ.*|CD>*<NN.*>+}
            {<PRP|WP>}
        PP: {<IN|TO>}
        VP: {<MD>?<VB.*>+}
        ADVP: {<RB.*>+}
        ADJP: {<JJ.*>+}
    ''')
    sentences = [nltk.word_tokenize(sentence, preserve_line=True) for sentence in nltk.sent_tokenize(text)]
    split = []
    for tagged in nltk.pos_tag_sents([tokens for tokens in sentences if tokens]):
        tokens, pos, chunks = zip(*tree2conlltags(chunker.parse(tagged)))
        split.append((tokens, pos, chunks, []))
    return split


def normalize_bio(labels):
    # Match the evaluator's span interpretation for an I-tag that starts a span.
    normalized = []
    previous_type = None
    for label in labels:
        entity_type = None if label == "O" else label[2:]
        if label.startswith("I-") and entity_type != previous_type:
            label = f"B-{entity_type}"
        normalized.append(label)
        previous_type = entity_type
    return normalized


def tag_text(text):
    if not models_ready:
        raise RuntimeError("Run the training cells with RUN_EXPERIMENTS = True first.")
    if not text.strip():
        raise ValueError("Enter some English text to tag.")

    split = prepare_text(text)
    table = pd.DataFrame([
        {"Sentence": sentence_number, "Token": token}
        for sentence_number, (tokens, _, _, _) in enumerate(split, start=1)
        for token in tokens
    ])
    for model_name, predictions in predict_all_models(split).items():
        table[model_name] = [label for sequence in predictions for label in normalize_bio(sequence)]
    return table

In [17]:
text_to_tag = """Ramy visits Spain."""
if models_ready:
    display(tag_text(text_to_tag))

,Sentence,Token,N-gram,Word2Vec,GloVe,LSTM + pretrained embeddings,LSTM + trainable embedding
0,1,Ramy,B-ORG,B-MISC,B-PER,B-PER,O
1,1,visits,O,B-PER,I-PER,O,O
2,1,Spain,B-LOC,O,B-LOC,B-LOC,B-LOC
3,1,.,O,O,O,O,O
